In [1]:
import pandas as pd
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [2]:
df = df.drop('customerID', axis=1)


In [3]:
df.columns


Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [4]:
df['gender'] = df['gender'].map({'Female': 0, 'Male': 1})
df['Partner'] = df['Partner'].map({'No': 0, 'Yes': 1})      
df['Dependents'] = df['Dependents'].map({'No': 0, 'Yes': 1})
df['PhoneService'] = df['PhoneService'].map({'No': 0, 'Yes': 1})
df['PaperlessBilling'] = df['PaperlessBilling'].map({'No': 0, 'Yes': 1})
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})


In [5]:
df.dtypes


gender                int64
SeniorCitizen         int64
Partner               int64
Dependents            int64
tenure                int64
PhoneService          int64
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling      int64
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

In [6]:
df=pd.get_dummies(df, columns=['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'], drop_first=True)


In [7]:
df.dtypes


gender                                     int64
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
MultipleLines_No phone service              bool
MultipleLines_Yes                           bool
InternetService_Fiber optic                 bool
InternetService_No                          bool
OnlineSecurity_No internet service          bool
OnlineSecurity_Yes                          bool
OnlineBackup_No internet service            bool
OnlineBackup_Yes                            bool
DeviceProtection_No internet service        bool
DeviceProtection_Yes                        bool
TechSupport_No inter

In [8]:
X = df.drop('Churn', axis=1)
y = df['Churn']


In [9]:
y = df['Churn']


In [10]:
from sklearn.model_selection import train_test_split


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [12]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)


Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

In [13]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()


In [14]:
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])


In [15]:
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])


In [16]:
X_train.shape, X_test.shape


((5634, 30), (1409, 30))

In [17]:
y_train.value_counts(normalize=True)


Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

In [18]:
y_test.value_counts(normalize=True)


Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

In [19]:
X_train[numeric_cols].describe()


,tenure,MonthlyCharges,TotalCharges
count,5.634000e+03,5.634000e+03,5.634000e+03
mean,-1.008935e-17,-2.402527e-16,2.522338e-17
std,1.000089e+00,1.000089e+00,1.000089e+00
min,-1.322329e+00,-1.544028e+00,-1.008922e+00
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01
50%,-1.418632e-01,1.848336e-01,-3.968446e-01
75%,9.164859e-01,8.319124e-01,6.741944e-01
max,1.608483e+00,1.785939e+00,2.801869e+00


In [20]:
addon_yes_cols = ['OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
                    'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes']
X_train['TotalAddons'] = X_train[addon_yes_cols].sum(axis=1)
X_test['TotalAddons'] = X_test[addon_yes_cols].sum(axis=1)


In [21]:
X_train['TotalAddons'].value_counts()
X_train['TotalAddons'].describe()


count    5634.000000
mean        2.058040
std         1.854652
min         0.000000
25%         0.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: TotalAddons, dtype: float64

In [22]:
import pandas as pd
pd.concat([X_train['TotalAddons'], y_train], axis=1).groupby('TotalAddons')['Churn'].mean()


TotalAddons
0    0.209448
1    0.458225
2    0.361779
3    0.277966
4    0.230216
5    0.125000
6    0.046809
Name: Churn, dtype: float64

TotalAddons validation:
Clear pattern for 1-6 addons: churn drops steadily as addon count rises
(46% churn at 1 addon → 4.7% churn at 6 addons) — strong real signal
Anomaly at 0 addons (20.9% churn, lower than 1-addon group) — likely because
"0" mixes two different populations: customers who chose zero addons vs
customers with no internet service at all (addons don't apply to them)
Feature is validated as useful overall; potential future refinement:
separate "no internet" from "internet but no addons chosen"

In [23]:
X_train[['tenure', 'MonthlyCharges', 'TotalCharges']].corr()


,tenure,MonthlyCharges,TotalCharges
tenure,1.000000,0.256700,0.829698
MonthlyCharges,0.256700,1.000000,0.654117
TotalCharges,0.829698,0.654117,1.000000


In [24]:
X_train.dtypes.value_counts()


bool       21
int64       7
float64     3
Name: count, dtype: int64

In [25]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)


In [26]:
X_train.dtypes.value_counts()


float64    31
Name: count, dtype: int64

In [27]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_data = pd.DataFrame()
vif_data['feature'] = X_train.columns
vif_data['VIF'] = [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])]
vif_data.sort_values('VIF', ascending=False)


c:\Users\TIAA USER\ml-notebook\venv\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
12,InternetService_No,inf
14,OnlineSecurity_Yes,inf
13,OnlineSecurity_No internet service,inf
18,DeviceProtection_Yes,inf
30,TotalAddons,inf
24,StreamingMovies_Yes,inf
23,StreamingMovies_No internet service,inf
22,StreamingTV_Yes,inf
21,StreamingTV_No internet service,inf
20,TechSupport_Yes,inf


In [28]:
redundant_cols = ['OnlineSecurity_No internet service', 'OnlineBackup_No internet service',
                   'DeviceProtection_No internet service', 'TechSupport_No internet service',
                   'StreamingTV_No internet service', 'StreamingMovies_No internet service']
X_train = X_train.drop(columns=redundant_cols)
X_test = X_test.drop(columns=redundant_cols)


In [29]:
addon_yes_cols = ['OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes',
                    'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes']
X_train = X_train.drop(columns=addon_yes_cols)
X_test = X_test.drop(columns=addon_yes_cols)


In [30]:
vif_data = pd.DataFrame()
vif_data['feature'] = X_train.columns
vif_data['VIF'] = [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])]
vif_data.sort_values('VIF', ascending=False)


,feature,VIF
7,MonthlyCharges,136.313677
5,PhoneService,82.232452
18,TotalAddons,28.490965
9,MultipleLines_No phone service,27.939098
11,InternetService_Fiber optic,27.792297
12,InternetService_No,16.140330
8,TotalCharges,10.968804
4,tenure,7.627556
14,Contract_Two year,2.599085
10,MultipleLines_Yes,2.473648


In [31]:
X_train = X_train.drop(columns=['MultipleLines_No phone service'])
X_test = X_test.drop(columns=['MultipleLines_No phone service'])


In [32]:
X_train = X_train.drop(columns=['TotalCharges'])
X_test = X_test.drop(columns=['TotalCharges'])


In [33]:
vif_data = pd.DataFrame()
vif_data['feature'] = X_train.columns
vif_data['VIF'] = [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])]
vif_data.sort_values('VIF', ascending=False)


,feature,VIF
7,MonthlyCharges,11.595051
5,PhoneService,9.707248
9,InternetService_Fiber optic,7.342117
16,TotalAddons,6.488581
10,InternetService_No,5.614565
12,Contract_Two year,3.392759
6,PaperlessBilling,2.861071
14,PaymentMethod_Electronic check,2.836946
2,Partner,2.800868
4,tenure,2.676890


Process:
Initial correlation check: tenure-TotalCharges (0.83), MonthlyCharges-TotalCharges (0.65)
First VIF run → inf values everywhere, revealed TWO hidden encoding-artifact traps:
a) "No internet service" duplicated identically across 6 service columns
b) TotalAddons double-counting its own raw component columns
Fixed both artifacts (dropped 6 redundant "No internet service" cols +
6 raw addon _Yes cols, kept TotalAddons)
Second VIF run → surfaced a THIRD hidden artifact: MultipleLines_No phone service
was redundant with PhoneService (same "impossible category" trap)
Dropped MultipleLines_No phone service and TotalCharges (the originally
planned drop, now confirmed VIF ~11 pre-fix)
Final VIF run: all features under/near threshold. MonthlyCharges (11.6) is
real business correlation (fiber optic genuinely costs more), not an artifact
— left as-is, to be handled via regularization if Logistic Regression needs it
Key lesson: one-hot encoding categorical columns that share a "not applicable"
category (like "No internet service") creates PERFECT artificial multicollinearity
— always check for this pattern specifically, not just generic high correlation.